# Decision Tree & Random Forest Fold Results Summary
This notebook summarizes the results of the `*fold_results.csv` files for different window sizes into a single Pandas DataFrame.

In [3]:
import os
import glob
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import filedialog

### Define Parser Function
We iterate over all files in the `DT` and `RF` directories to extract string definitions and means/standard deviations.

In [4]:
def process_directory(base_dir):
    # Dynamically find all model folders rather than assuming RF/DT
    possible_models = ['DT', 'RF', 'GB', 'XGB', 'LR', 'SVM']
    models = [m for m in possible_models if os.path.isdir(os.path.join(base_dir, m))]
    
    if not models:
        # Fallback to scanning everything if they used customized names
        dirs = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
        models = [d for d in dirs if d in possible_models]
        
    if not models:
        print("No recognized model directories found in", base_dir)
        return pd.DataFrame()
        
    results = {}
    
    for model in models:
        pattern = os.path.join(base_dir, model, '*', '*fold_results.csv')
        files = glob.glob(pattern)
        
        for file in files:
            filename = os.path.basename(file)
            parts = filename.split('_')
            
            # Extract Dataset Name and Window dynamically
            window = ""
            dataset_name = parts[0] + "_Features"
            
            for i, p in enumerate(parts):
                if p.endswith('s') and p[:-1].isdigit():
                    window = p
                    if i > 0:
                        dataset_name = "_".join(parts[:i])
                    break
            
            df = pd.read_csv(file)
            metrics = ['balanced_accuracy', 'f1_score', 'sensitivity', 'specificity']
            stats = {}
            for m in metrics:
                if m in df.columns:
                    mean_val = df[m].mean()
                    std_val = df[m].std()
                    stats[m] = f"{mean_val:.4f} ± {std_val:.4f}"
                else:
                    stats[m] = "N/A"
                
            features_used = df['features_used'].iloc[0] if 'features_used' in df.columns else "Dynamic Extraction"
            
            key = (dataset_name, window)
            if key not in results:
                results[key] = {'dataset': dataset_name, 'window': window, 'features': features_used}
            
            for m in metrics:
                results[key][f"{m}_{model}"] = stats[m]
                
    # Build dataframe rows dynamically
    rows = []
    for (ds, w), row_data in results.items():
        row = [row_data.get('dataset', ds), row_data.get('window', "")]
        for model in models:
            row.append(row_data.get(f'balanced_accuracy_{model}', ""))
            row.append(row_data.get(f'f1_score_{model}', ""))
            row.append(row_data.get(f'sensitivity_{model}', ""))
            row.append(row_data.get(f'specificity_{model}', ""))
        
        row.append(row_data.get('features', ""))
        row.append("Random undersampling (Auto)") 
        rows.append(row)
        
    # Sort windows
    def window_sort_key(r):
        w = str(r[1]).replace('s','')
        return int(w) if w.isdigit() else 999
        
    rows.sort(key=window_sort_key)
    
    cols = ["dataset", "window"]
    for model in models:
        cols.extend([f"balanced_accuracy_{model}", f"f1_score_{model}", f"sensitivity_{model}", f"specificity_{model}"])
    cols.extend(["feature list", "imbalance handeling"])
    
    out_df = pd.DataFrame(rows, columns=cols)
    return out_df

### Execute and Save
Run the function on the current directory and display the summary.

In [ ]:
# Launch Tkinter Directory Picker for Modularity.
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)
root.update()

current_dir = filedialog.askdirectory(title="Select extracted_features Directory")

root.update()
root.destroy()

if current_dir:
    print(f"Processing directory: {current_dir}")
    summary_df = process_directory(current_dir)
    
    if not summary_df.empty:
        display(summary_df)

        # Save to CSV using the parent folder explicitly
        csv_path = os.path.join(current_dir, 'model_summary_table.csv')
        summary_df.to_csv(csv_path, index=False)
        print(f"\nSaved dynamic summary to: \n{csv_path}")
    else:
        print("No fold result data found.")
else:
    print("No directory selected.")

2026-03-20 11:05:58.758 python[69415:1125571] The class 'NSOpenPanel' overrides the method identifier.  This method is implemented by class 'NSWindow'


Processing directory: /Volumes/ss/Project_CareWear/DATASET/ss_drive/GalaxyPPG/5_activity_chunks/GalaxyWatch/hr_chunks/extracted_features


,dataset,window,balanced_accuracy_DT,f1_score_DT,sensitivity_DT,specificity_DT,balanced_accuracy_RF,f1_score_RF,sensitivity_RF,specificity_RF,...,balanced_accuracy_LR,f1_score_LR,sensitivity_LR,specificity_LR,balanced_accuracy_SVM,f1_score_SVM,sensitivity_SVM,specificity_SVM,feature list,imbalance handeling
0,GalaxyPPG_Features,2s,0.4611 ± 0.1415,0.2021 ± 0.1071,0.4611 ± 0.1415,0.6907 ± 0.0338,0.4571 ± 0.1383,0.2017 ± 0.1070,0.4571 ± 0.1383,0.6905 ± 0.0337,...,0.4604 ± 0.1409,0.2064 ± 0.1088,0.4604 ± 0.1409,0.6900 ± 0.0318,0.4764 ± 0.1223,0.2039 ± 0.1068,0.4764 ± 0.1223,0.6921 ± 0.0335,"HR Range: 0–40 bpm, HR Range: 40–60 bpm, HR Ra...",Random undersampling (Auto)
1,GalaxyPPG_Features,5s,0.4800 ± 0.1199,0.2254 ± 0.1119,0.4800 ± 0.1199,0.6892 ± 0.0292,0.4819 ± 0.1187,0.2256 ± 0.1084,0.4819 ± 0.1187,0.6902 ± 0.0298,...,0.4831 ± 0.1256,0.2204 ± 0.1118,0.4831 ± 0.1256,0.6930 ± 0.0322,0.4708 ± 0.1404,0.2236 ± 0.1125,0.4708 ± 0.1404,0.6912 ± 0.0324,"HR Range: 0–40 bpm, HR Range: 40–60 bpm, HR Ra...",Random undersampling (Auto)
2,GalaxyPPG_Features,10s,0.4705 ± 0.1456,0.2474 ± 0.1050,0.4705 ± 0.1456,0.6920 ± 0.0278,0.4783 ± 0.1361,0.2562 ± 0.1044,0.4783 ± 0.1361,0.6925 ± 0.0269,...,0.4902 ± 0.1181,0.2281 ± 0.1116,0.4902 ± 0.1181,0.6937 ± 0.0330,0.4880 ± 0.1202,0.2218 ± 0.1169,0.4880 ± 0.1202,0.6928 ± 0.0358,"HR Range: 0–40 bpm, HR Range: 40–60 bpm, HR Ra...",Random undersampling (Auto)
3,GalaxyPPG_Features,30s,0.4301 ± 0.1836,0.2907 ± 0.1449,0.4301 ± 0.1836,0.6782 ± 0.0776,0.4646 ± 0.1503,0.2876 ± 0.1475,0.4646 ± 0.1503,0.6746 ± 0.0755,...,0.4880 ± 0.1355,0.2523 ± 0.1214,0.4810 ± 0.1417,0.6955 ± 0.0361,0.4700 ± 0.1297,0.2646 ± 0.1683,0.4700 ± 0.1297,0.6917 ± 0.0494,"HR Range: 0–40 bpm, HR Range: 40–60 bpm, HR Ra...",Random undersampling (Auto)
4,GalaxyPPG_Features,60s,0.5236 ± 0.0875,0.4793 ± 0.1085,0.5236 ± 0.0875,0.5236 ± 0.0875,0.5284 ± 0.1029,0.4926 ± 0.1203,0.5284 ± 0.1029,0.5284 ± 0.1029,...,0.5510 ± 0.1106,0.4953 ± 0.1377,0.5510 ± 0.1106,0.5510 ± 0.1106,0.5532 ± 0.1094,0.4940 ± 0.1402,0.5532 ± 0.1094,0.5532 ± 0.1094,"HR Range: 0–40 bpm, HR Range: 40–60 bpm, HR Ra...",Random undersampling (Auto)



Saved dynamic summary to: 
/Volumes/ss/Project_CareWear/DATASET/ss_drive/GalaxyPPG/5_activity_chunks/GalaxyWatch/hr_chunks/extracted_features/model_summary_table.csv


: 